# 🏃 App 1 — Tạo index khuôn mặt & số BIB (Colab/GPU)

**Cách dùng:**
1. **Runtime → Change runtime type → GPU**
2. Upload file `app1_indexer.zip` khi được hỏi
3. Chạy cả 3 cell → link giao diện hiện ra (dạng `https://xxxxx.gradio.live`)
4. Mở link → dán link Drive → bấm chạy → tự tải index về

## 1. Upload code & cài thư viện (5–7 phút lần đầu)

In [ ]:
import os, zipfile

WORK   = '/content/app1_indexer'
MARKER = '/content/.deps_installed_v7'

# --- Upload và giải nén zip code ---
if not os.path.exists(WORK):
    from google.colab import files
    print('📦 Hãy upload file app1_indexer.zip:')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall('/content')
    os.remove(zip_name)
    if not os.path.exists(WORK):
        extracted = [d for d in os.listdir('/content')
                     if os.path.isdir(f'/content/{d}') and d not in
                     ('sample_data', '.config', '.local')]
        for d in extracted:
            if os.path.exists(f'/content/{d}/build_index.py'):
                os.rename(f'/content/{d}', WORK)
                break
        else:
            os.makedirs(WORK, exist_ok=True)
            for f in ('build_index.py', 'face_engine.py', 'bib_engine.py',
                      'config.py', 'drive_source.py', 'bib_detector.py'):
                if os.path.exists(f'/content/{f}'):
                    os.rename(f'/content/{f}', f'{WORK}/{f}')

os.chdir(WORK)

NEEDED = ['build_index.py', 'face_engine.py', 'bib_engine.py',
          'config.py', 'drive_source.py', 'bib_detector.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f'Thiếu file: {missing}'
print('✅ Code sẵn sàng tại:', os.getcwd())

# --- Cài deps ---
if os.path.exists(MARKER):
    print('✅ Thư viện đã cài từ trước (skip).')
else:
    # google-auth và google-api-python-client: dùng bản Colab có sẵn (không pin)
    !pip install -q \
      "numpy==1.26.4" \
      insightface==0.7.3 onnxruntime-gpu==1.19.2 faiss-cpu==1.8.0 \
      paddleocr==2.7.3 paddlepaddle==2.6.2 ultralytics==8.2.0 \
      opencv-python-headless==4.10.0.84 pandas==2.2.2 pyarrow==16.1.0 \
      python-dotenv==1.0.1 tqdm==4.66.4 gradio==4.44.1
    !pip install -q --force-reinstall --no-deps "numpy==1.26.4"
    open(MARKER, 'w').close()
    print('\n⚠️  Đã cài xong. Đang restart runtime...')
    os.kill(os.getpid(), 9)

!nvidia-smi -L
print('✅ Sẵn sàng — chạy cell tiếp theo.')

## 2. Đăng nhập Google Drive

In [ ]:
import os, sys

WORK = '/content/app1_indexer'
if os.getcwd() != WORK:
    os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

from google.colab import auth
auth.authenticate_user()
from google.auth import default
creds, _ = default(scopes=['https://www.googleapis.com/auth/drive'])

os.environ.setdefault('FACE_CTX_ID', '0')
os.environ.setdefault('ENABLE_BIB', 'true')
os.environ.setdefault('DRIVE_FOLDER_ID', 'placeholder')  # sẽ set đúng khi chạy

from drive_source import DriveSource
drive = DriveSource(credentials=creds)
print('✅ Đã đăng nhập Google Drive — chạy cell tiếp để mở giao diện.')

## 3. Mở giao diện quét (Gradio — link riêng)

In [ ]:
import re, shutil, time, traceback, os, sys
import gradio as gr
from googleapiclient.http import MediaFileUpload

WORK = '/content/app1_indexer'
if os.getcwd() != WORK:
    os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

from config import IndexerConfig
from build_index import build_index


def _extract_folder_id(text: str) -> str | None:
    """Trích folder ID từ link Drive hoặc ID thuần."""
    text = text.strip()
    if not text:
        return None
    m = re.search(r'folders/([a-zA-Z0-9_-]+)', text)
    if m:
        return m.group(1)
    if re.fullmatch(r'[a-zA-Z0-9_-]{10,}', text):
        return text
    return None


def _upload_to_drive(zip_path: str, folder_id_short: str) -> str:
    """Upload file zip lên root Drive."""
    service = drive._service
    file_metadata = {'name': f'index_{folder_id_short}.zip', 'mimeType': 'application/zip'}
    media = MediaFileUpload(zip_path, mimetype='application/zip', resumable=True)
    uploaded = service.files().create(
        body=file_metadata, media_body=media, fields='id,webViewLink'
    ).execute()
    return uploaded.get('webViewLink', f"ID: {uploaded['id']}")


def run_indexer(links_text: str, enable_bib: bool, save_drive: bool,
               limit: int, progress=gr.Progress(track_tqdm=True)):
    """Xử lý danh sách folder Drive."""
    lines = links_text.strip().splitlines()
    folder_ids = [fid for line in lines if (fid := _extract_folder_id(line))]

    if not folder_ids:
        return "❌ Không tìm thấy folder ID hợp lệ. Dán link hoặc ID vào ô trên.", None

    os.environ['ENABLE_BIB'] = 'true' if enable_bib else 'false'
    actual_limit = limit if limit > 0 else None

    log_lines: list[str] = []
    output_files: list[str] = []

    for i, fid in enumerate(folder_ids, 1):
        log_lines.append(f"{'='*50}")
        log_lines.append(f"▶ [{i}/{len(folder_ids)}] Folder: {fid}")
        log_lines.append(f"{'='*50}")

        os.environ['DRIVE_FOLDER_ID'] = fid
        output_dir = f'/content/output_{fid[:12]}'
        os.environ['INDEX_OUTPUT_DIR'] = output_dir

        try:
            config = IndexerConfig.from_env()
            t0 = time.time()
            build_index(config, limit=actual_limit, drive=drive)
            elapsed = time.time() - t0
            log_lines.append(f"⏱️  Hoàn thành trong {elapsed:.0f}s")

            # Tạo zip
            zip_path = f'/content/index_{fid[:12]}'
            shutil.make_archive(zip_path, 'zip', output_dir)
            zip_file = zip_path + '.zip'
            size_mb = os.path.getsize(zip_file) / (1024 * 1024)
            log_lines.append(f"📦 {zip_file} ({size_mb:.1f} MB)")
            output_files.append(zip_file)

            # Upload lên Drive
            if save_drive:
                log_lines.append("☁️  Đang upload lên Drive...")
                link = _upload_to_drive(zip_file, fid[:12])
                log_lines.append(f"✅ Đã upload: {link}")

        except Exception:
            log_lines.append(f"❌ Lỗi: {traceback.format_exc()}")
            continue

        log_lines.append("")

    log_lines.append(f"\n🎉 Xong {len(output_files)}/{len(folder_ids)} folder!")
    log_text = "\n".join(log_lines)

    # Trả file đầu tiên để tải (Gradio chỉ hỗ trợ 1 file output đơn giản)
    # Nếu nhiều file, zip lại thành 1
    if len(output_files) == 0:
        return log_text, None
    elif len(output_files) == 1:
        return log_text, output_files[0]
    else:
        combined = '/content/all_indexes.zip'
        import zipfile
        with zipfile.ZipFile(combined, 'w') as zf:
            for f in output_files:
                zf.write(f, os.path.basename(f))
        return log_text, combined


# ─── Giao diện Gradio ──────────────────────────────────────────────────────────

with gr.Blocks(title="🏃 Face Indexer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🏃 Tạo index khuôn mặt & số BIB từ Google Drive")

    with gr.Row():
        txt_links = gr.Textbox(
            label="Link / ID folder Drive (mỗi dòng 1 folder)",
            placeholder="https://drive.google.com/drive/folders/1ABC...\nhoặc chỉ dán ID: 1ABC...",
            lines=5,
        )

    with gr.Row():
        chk_bib = gr.Checkbox(label="Đọc số BIB", value=True)
        chk_drive = gr.Checkbox(label="Lưu index lên Drive", value=True)
        num_limit = gr.Number(label="Giới hạn ảnh (0 = toàn bộ)", value=0, precision=0)

    btn_start = gr.Button("🚀 Bắt đầu quét", variant="primary", size="lg")

    txt_log = gr.Textbox(label="Log", lines=15, interactive=False)
    file_out = gr.File(label="📥 Tải index về máy")

    btn_start.click(
        fn=run_indexer,
        inputs=[txt_links, chk_bib, chk_drive, num_limit],
        outputs=[txt_log, file_out],
    )

demo.launch(share=True, debug=False)
print('\n⬆️  Mở link phía trên để sử dụng giao diện.')